# Model training HITL ANNOTATION

In this python notebook I will cross validate 3 models, one on manual annotated data, one on grabcut annotated data, and a final one on SAM2 annotated data.

In [ ]:
from ultralytics import YOLO

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

Here we cross validate the manually annotated data, so we train with manual and validate with manual

In [ ]:
fold_yamls_manual = [f"Fold{i+1}MANUAL.yaml" for i in range(5)]
print(fold_yamls_manual)

In [ ]:
for i, yaml_file in enumerate(fold_yamls_manual):
    # Load a fresh YOLO26 Nano model for each fold
    model = YOLO("testRun.pt")

    # Train the model, saving results to a unique project directory
    results = model.train(data=yaml_file, epochs=20, project="cv_experiment", name=f"fold{i+1}Manual", batch=8, imgsz=640)

Here we cross-validate the grabcut annotated data, using grabcut data to train and manual data to validate 

In [ ]:
fold_yamls_grabcut = [f"Fold{i+1}GRABCUT.yaml" for i in range(5)]
print(fold_yamls_grabcut)

In [ ]:
for i, yaml_file in enumerate(fold_yamls_grabcut):
    # Load a fresh YOLO26 Nano model for each fold
    model = YOLO("testRun.pt")

    # Train the model, saving results to a unique project directory
    results = model.train(data=yaml_file, epochs=20, project="cv_experiment", name=f"fold{i+1}Grabcut", batch=8, imgsz=640)

Here we cross-validate the SAM2 annotated data, using SAM2 data to train and manual data to validate

In [ ]:
fold_yamls_SAM2 = [f"Fold{i+1}SAM2.yaml" for i in range(5)]
print(fold_yamls_SAM2)

In [ ]:
for i, yaml_file in enumerate(fold_yamls_SAM2):
    # Load a fresh YOLO26 Nano model for each fold
    model = YOLO("testRun.pt")

    # Train the model, saving results to a unique project directory
    results = model.train(data=yaml_file, epochs=20, project="cv_experiment", name=f"fold{i+1}SAM2", batch=8, imgsz=640)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from scipy.spatial import distance
import matplotlib.pyplot as plt

We now extract the predicted masks from the manual annotation trained models.

In [ ]:
for i in range(5):
    model = YOLO(f"runs/semantic/cv_experiment/fold{i+1}Manual/weights/best.pt")
    predictions = model.predict(source=f"crossValidationFolder/Fold{i+1}/datasetManual/images/val")


    for predict in predictions:
        img_name = os.path.basename(predict.path)
    
        # Get the full dense class map (H, W)
        class_map = predict.semantic_mask.data
    
        # Create a binary mask for a specific class (e.g., class 0)
        binary_mask = (class_map == 1).cpu().numpy()
        binary_mask = (binary_mask * 255).astype('uint8')
    
    
        save_path = 'predictedMasks\\predictedManual\\'+img_name
        print(save_path)
        success = cv2.imwrite(save_path, binary_mask)
        if not success:
            print("Something went wrong!")

We now extract the predicted masks for grabcut annotation trained models.

In [ ]:
for i in range(5):
    model = YOLO(f"runs/semantic/cv_experiment/fold{i+1}Grabcut/weights/best.pt")
    predictions = model.predict(source=f"crossValidationFolder/Fold{i+1}/datasetManual/images/val")


    for predict in predictions:
        img_name = os.path.basename(predict.path)
    
        # Get the full dense class map (H, W)
        class_map = predict.semantic_mask.data
    
        # Create a binary mask for a specific class (e.g., class 0)
        binary_mask = (class_map == 1).cpu().numpy()
        binary_mask = (binary_mask * 255).astype('uint8')
    
    
        save_path = 'predictedMasks\\predictedGrabcut\\'+img_name
        print(save_path)
        success = cv2.imwrite(save_path, binary_mask)
        if not success:
            print("Something went wrong!")

We now extract the masks for the SAM2 annotation trained models.

In [ ]:
for i in range(5):
    model = YOLO(f"runs/semantic/cv_experiment/fold{i+1}SAM2/weights/best.pt")
    predictions = model.predict(source=f"crossValidationFolder/Fold{i+1}/datasetManual/images/val")


    for predict in predictions:
        img_name = os.path.basename(predict.path)
    
        # Get the full dense class map (H, W)
        class_map = predict.semantic_mask.data
    
        # Create a binary mask for a specific class (e.g., class 0)
        binary_mask = (class_map == 1).cpu().numpy()
        binary_mask = (binary_mask * 255).astype('uint8')
    
    
        save_path = 'predictedMasks\\predictedSAM2\\'+img_name
        print(save_path)
        success = cv2.imwrite(save_path, binary_mask)
        if not success:
            print("Something went wrong!")

We now compare the pixel area and Ferret size of all the masks with the original manual annotation.

In [ ]:
def get_mask_metrics(img_path):
    """Calculates pixel area, true max Feret length, and true min Feret length 
    for a binary mask using the strict parallel-caliper geometric definition.
    """
    # Load as grayscale
    mask = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    # Calculate Pixel Area
    area = np.sum(mask > 0)
    
    max_feret = 0.0
    min_feret = 0.0
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    if contours:
        # Get the largest contour to avoid counting random background noise
        largest_contour = max(contours, key=cv2.contourArea)
        
        # We use the Convex Hull because Feret diameters only depend on extreme boundary points
        hull = cv2.convexHull(largest_contour)
        hull_points = hull.reshape(-1, 2)
        num_points = hull_points.shape[0]
        
        if num_points > 2:
            # Max distance between any two antipodal points on the hull
            dist_matrix = distance.cdist(hull_points, hull_points, 'euclidean')
            max_feret = np.max(dist_matrix)
            
            min_caliper_width = float('inf')
            
            for i in range(num_points):
                # Define the current caliper line passing through edge (P_i to P_{i+1})
                p1 = hull_points[i]
                p2 = hull_points[(i + 1) % num_points]
                
                # Edge vector
                edge = p2 - p1
                edge_len = np.linalg.norm(edge)
                
                if edge_len == 0:
                    continue
                
                # Calculate the perpendicular distance from ALL other hull points to this edge line
                cross_product = np.abs(np.cross(edge, p1 - hull_points))
                distances = cross_product / edge_len
                
                # The caliper width for this specific parallel alignment is the max distance found
                max_distance_for_edge = np.max(distances)
                
                if max_distance_for_edge < min_caliper_width:
                    min_caliper_width = max_distance_for_edge
            
            min_feret = min_caliper_width
            
        elif num_points == 2:
            # The object is a straight line segment
            d = np.linalg.norm(hull_points[0] - hull_points[1])
            max_feret = d
            min_feret = 0.0 # A perfect line has 0 thickness perpendicular to it
        elif num_points == 1:
            # Single pixel object
            max_feret = 1.0
            min_feret = 1.0
        
    return int(area), float(max_feret), float(min_feret)

We compute the pixel area, the maximum feret size, and minimum feret size for the predictions from manual annotation trained models.

In [ ]:
results = []
PREDICTIONS_DIR = r'predictedMasks\predictedManual'
ANNOTATIONS_DIR = r'dataset\CleanManualAnnotations'

for img_name in os.listdir(PREDICTIONS_DIR):
    pred_path = os.path.join(PREDICTIONS_DIR, img_name)
    annot_path = os.path.join(ANNOTATIONS_DIR, img_name)
    
    if os.path.isfile(pred_path) and os.path.isfile(annot_path):
        pred_area, pred_max_f, pred_min_f = get_mask_metrics(pred_path)
        annot_area, annot_max_f, annot_min_f = get_mask_metrics(annot_path)
        
        if pred_area is not None and annot_area is not None:
            # Calculate Absolute Differences
            area_diff = pred_area - annot_area
            max_f_diff = pred_max_f - annot_max_f
            min_f_diff = pred_min_f - annot_min_f
            
            # Calculate Percentage Differences (relative to annotation baseline)
            area_pct_diff = (area_diff / annot_area * 100) if annot_area > 0 else 0
            max_f_pct_diff = (max_f_diff / annot_max_f * 100) if annot_max_f > 0 else 0
            min_f_pct_diff = (min_f_diff / annot_min_f * 100) if annot_min_f > 0 else 0
            
            results.append({
                'Filename': img_name,
                
                # Area Comparison
                'Annot_Area': annot_area,
                'Pred_Area': pred_area,
                'Area_Diff': area_diff,
                'Area_Pct_Diff (%)': round(area_pct_diff, 2),
                
                # Maximum Feret Comparison
                'Annot_Max_Feret': round(annot_max_f, 2),
                'Pred_Max_Feret': round(pred_max_f, 2),
                'Max_Feret_Diff': round(max_f_diff, 2),
                'Max_Feret_Pct_Diff (%)': round(max_f_pct_diff, 2),
                
                # Minimum Feret Comparison
                'Annot_Min_Feret': round(annot_min_f, 2),
                'Pred_Min_Feret': round(pred_min_f, 2),
                'Min_Feret_Diff': round(min_f_diff, 2),
                'Min_Feret_Pct_Diff (%)': round(min_f_pct_diff, 2)
            })

# Compile results into a Pandas DataFrame
df = pd.DataFrame(results)
print(df)
df.to_csv("CSVoutputs/ManualMaskStatsFIXED.csv", index=False)

We compute the pixel area, the maximum feret size, and minimum feret size for the predictions from grabcut annotation trained models.


In [ ]:
results = []
PREDICTIONS_DIR = r'predictedMasks\predictedGrabcut'
ANNOTATIONS_DIR = r'dataset\CleanManualAnnotations'

for img_name in os.listdir(PREDICTIONS_DIR):
    pred_path = os.path.join(PREDICTIONS_DIR, img_name)
    annot_path = os.path.join(ANNOTATIONS_DIR, img_name)
    
    if os.path.isfile(pred_path) and os.path.isfile(annot_path):
        pred_area, pred_max_f, pred_min_f = get_mask_metrics(pred_path)
        annot_area, annot_max_f, annot_min_f = get_mask_metrics(annot_path)
        
        if pred_area is not None and annot_area is not None:
            # Calculate Absolute Differences
            area_diff = pred_area - annot_area
            max_f_diff = pred_max_f - annot_max_f
            min_f_diff = pred_min_f - annot_min_f
            
            # Calculate Percentage Differences (relative to annotation baseline)
            area_pct_diff = (area_diff / annot_area * 100) if annot_area > 0 else 0
            max_f_pct_diff = (max_f_diff / annot_max_f * 100) if annot_max_f > 0 else 0
            min_f_pct_diff = (min_f_diff / annot_min_f * 100) if annot_min_f > 0 else 0
            
            results.append({
                'Filename': img_name,
                
                # Area Comparison
                'Annot_Area': annot_area,
                'Pred_Area': pred_area,
                'Area_Diff': area_diff,
                'Area_Pct_Diff (%)': round(area_pct_diff, 2),
                
                # Maximum Feret Comparison
                'Annot_Max_Feret': round(annot_max_f, 2),
                'Pred_Max_Feret': round(pred_max_f, 2),
                'Max_Feret_Diff': round(max_f_diff, 2),
                'Max_Feret_Pct_Diff (%)': round(max_f_pct_diff, 2),
                
                # Minimum Feret Comparison
                'Annot_Min_Feret': round(annot_min_f, 2),
                'Pred_Min_Feret': round(pred_min_f, 2),
                'Min_Feret_Diff': round(min_f_diff, 2),
                'Min_Feret_Pct_Diff (%)': round(min_f_pct_diff, 2)
            })

# Compile results into a Pandas DataFrame
df = pd.DataFrame(results)
print(df)
df.to_csv("CSVoutputs/GrabcutMaskStatsFIXED.csv", index=False)

We compute the pixel area, the maximum feret size, and minimum feret size for the predictions from SAM2 annotation trained models.


In [ ]:
results = []
PREDICTIONS_DIR = r'predictedMasks\predictedSAM2'
ANNOTATIONS_DIR = r'dataset\CleanManualAnnotations'

for img_name in os.listdir(PREDICTIONS_DIR):
    pred_path = os.path.join(PREDICTIONS_DIR, img_name)
    annot_path = os.path.join(ANNOTATIONS_DIR, img_name)
    
    if os.path.isfile(pred_path) and os.path.isfile(annot_path):
        pred_area, pred_max_f, pred_min_f = get_mask_metrics(pred_path)
        annot_area, annot_max_f, annot_min_f = get_mask_metrics(annot_path)
        
        if pred_area is not None and annot_area is not None:
            # Calculate Absolute Differences
            area_diff = pred_area - annot_area
            max_f_diff = pred_max_f - annot_max_f
            min_f_diff = pred_min_f - annot_min_f
            
            # Calculate Percentage Differences (relative to annotation baseline)
            area_pct_diff = (area_diff / annot_area * 100) if annot_area > 0 else 0
            max_f_pct_diff = (max_f_diff / annot_max_f * 100) if annot_max_f > 0 else 0
            min_f_pct_diff = (min_f_diff / annot_min_f * 100) if annot_min_f > 0 else 0
            
            results.append({
                'Filename': img_name,
                
                # Area Comparison
                'Annot_Area': annot_area,
                'Pred_Area': pred_area,
                'Area_Diff': area_diff,
                'Area_Pct_Diff (%)': round(area_pct_diff, 2),
                
                # Maximum Feret Comparison
                'Annot_Max_Feret': round(annot_max_f, 2),
                'Pred_Max_Feret': round(pred_max_f, 2),
                'Max_Feret_Diff': round(max_f_diff, 2),
                'Max_Feret_Pct_Diff (%)': round(max_f_pct_diff, 2),
                
                # Minimum Feret Comparison
                'Annot_Min_Feret': round(annot_min_f, 2),
                'Pred_Min_Feret': round(pred_min_f, 2),
                'Min_Feret_Diff': round(min_f_diff, 2),
                'Min_Feret_Pct_Diff (%)': round(min_f_pct_diff, 2)
            })

# Compile results into a Pandas DataFrame
df = pd.DataFrame(results)
print(df)
df.to_csv("CSVoutputs/SAM2MaskStatsFIXED.csv", index=False)

From here on is some code to make metric visualizations for the report. This code is extremely quick and dirty as I do not need exact measures to get the idea across.

In [ ]:
def visualize_feret_metrics(img_path):
    """Loads a mask, calculates true geometric Feret metrics, 
    and draws accurate caliper-based visualizations.
    """
    # Load as grayscale and isolate object
    mask = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)    
    _, binary_mask = cv2.threshold(mask, 100, 255, cv2.THRESH_BINARY)
    cv2.rectangle(binary_mask, (0, 0), (binary_mask.shape[1]-1, binary_mask.shape[0]-1), 0, 1)
    
    vis_img = cv2.cvtColor(binary_mask, cv2.COLOR_GRAY2RGB)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    largest_contour = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(largest_contour)
    
    hull_points = hull.reshape(-1, 2)
    num_points = hull_points.shape[0]
    
    # Draw the convex hull outline in subtle blue
    cv2.drawContours(vis_img, [hull], -1, (255, 100, 50), 1)
    
    if num_points > 2:
        # --- 1. Max Feret Distance (Green Line) ---
        dist_matrix = distance.cdist(hull_points, hull_points, 'euclidean')
        max_idx = np.unravel_index(np.argmax(dist_matrix), dist_matrix.shape)
        pt1 = tuple(hull_points[max_idx[0]].astype(int))
        pt2 = tuple(hull_points[max_idx[1]].astype(int))
        cv2.line(vis_img, pt1, pt2, (0, 255, 0), 2) 
        
        # --- 2. Min Feret Distance (Magenta Caliper Measurement) ---
        min_caliper_width = float('inf')
        best_edge_pts = None
        best_antipodal_pt = None
        best_perp_vector = None
        
        for i in range(num_points):
            p1 = hull_points[i]
            p2 = hull_points[(i + 1) % num_points]
            
            edge = p2 - p1
            edge_len = np.linalg.norm(edge)
            if edge_len == 0:
                continue
                
            # Distance from all points to this specific edge line
            cross_product = np.cross(edge, p1 - hull_points)
            distances = np.abs(cross_product) / edge_len
            
            max_idx = np.argmax(distances)
            max_distance_for_edge = distances[max_idx]
            
            if max_distance_for_edge < min_caliper_width:
                min_caliper_width = max_distance_for_edge
                best_edge_pts = (p1, p2)
                best_antipodal_pt = hull_points[max_idx]
                
                # Calculate normal unit vector pointing from edge toward the antipodal point
                # Perpendicular direction to edge vector (-y, x)
                normal = np.array([-edge[1], edge[0]])
                normal = normal / np.linalg.norm(normal)
                
                # Verify direction matches the sign of cross product to face the correct way
                if cross_product[max_idx] < 0:
                    normal = -normal
                best_perp_vector = normal

        # Draw Min Feret Geometry
        if best_edge_pts is not None:
            e1, e2 = best_edge_pts
            
            # Find the projection point of the antipodal vertex onto the base edge line
            # Vector from e1 to antipodal point
            v = best_antipodal_pt - e1
            edge_vec = e2 - e1
            edge_unit = edge_vec / np.linalg.norm(edge_vec)
            projection_length = np.dot(v, edge_unit)
            
            # Point on the edge line exactly perpendicular to the antipodal point
            base_projected_pt = e1 + projection_length * edge_unit
            
            # Convert to integers for drawing
            p_start = tuple(base_projected_pt.astype(int))
            p_end = tuple(best_antipodal_pt.astype(int))
            
            # Draw the base caliper edge reference line in light gray
            cv2.line(vis_img, tuple(e1.astype(int)), tuple(e2.astype(int)), (180, 180, 180), 1)
            
            # Draw the true Min Feret thickness line (Magenta) across the span of the object
            cv2.line(vis_img, p_start, p_end, (255, 0, 255), 2)
            
    elif num_points == 2:
        # If it's just a line, Max Feret is the line length, Min is 0
        pt1 = tuple(hull_points[0].astype(int))
        pt2 = tuple(hull_points[1].astype(int))
        cv2.line(vis_img, pt1, pt2, (0, 255, 0), 2)
        
    return vis_img

In [ ]:
plt.imshow(visualize_feret_metrics("predictedMasks/predictedManual/07-11-segment8_camera1_slot_25_frame_0619-0.png"))

In [ ]:
plt.imshow(visualize_feret_metrics("dataset/CleanManualAnnotations/07-11-segment8_camera1_slot_25_frame_0619-0.png"))

Predicting some random images for fun

In [ ]:
model.predict("crossValidationFolder/Fold5/datasetManual/images/val/07-23-segment6_camera1_slot_8_frame_0246-0.png", save=True, device="cpu")


In [ ]:
print(i+1)